# Reserve 1: Saros-Grid-SSM - Google Colab Runner
**Project:** Solar/Wind Spatio-Temporal SSM for Spanish REE Electricity Grid  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Can continuous-time selective state space recurrence coupled with graph spatial diffusion predict 15-minute nodal renewable injections with wMAPE < 6.5% and curtailment F1 >= 0.75 without $O(N^2)$ attention?

Configured for Google Colab (Free T4 or Pro A100/L4) and local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB")
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'saros-grid-ssm'
    if not os.path.exists(repo_dir):
        print('Cloning saros-grid-ssm (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/saros-grid-ssm.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication (with 90s Idle-Kill Guard)
# Securely reads from Colab Secrets (Key icon on left sidebar) or reuses ~/.netrc.
# If interactive input hangs for > 90 seconds, automatically unassigns GPU to save credits.
import os
import signal
import sys
import wandb

def auth_timeout_handler(signum, frame):
    print("\n[Credit Guard Alert] W&B authentication timed out (90s elapsed without input)!")
    print("[Credit Guard Alert] Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
    raise TimeoutError("Interactive authentication timed out after 90 seconds to protect credits.")

if hasattr(signal, 'SIGALRM'):
    signal.signal(signal.SIGALRM, auth_timeout_handler)
    signal.alarm(90)

try:
    # 1. Attempt reading from Google Colab Secrets across candidate names
    try:
        from google.colab import userdata
        for secret_name in ['WANDB_API_KEY', 'wandb_api_key', 'WANDB', 'WAN', 'wandb']:
            try:
                val = userdata.get(secret_name)
                if val and len(val.strip()) > 10:
                    os.environ['WANDB_API_KEY'] = val.strip()
                    print(f"[W&B] Loaded API key from Colab Secrets ('{secret_name}').")
                    break
            except Exception:
                pass
    except Exception:
        pass

    # 2. Fallback: check ~/.netrc if previously logged in
    netrc_path = os.path.expanduser('~/.netrc')
    if 'WANDB_API_KEY' not in os.environ and os.path.exists(netrc_path):
        try:
            with open(netrc_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2 and parts[0] == 'password' and len(parts[1]) > 10:
                        os.environ['WANDB_API_KEY'] = parts[1]
                        print("[W&B] Restored WANDB_API_KEY automatically from ~/.netrc.")
                        break
        except Exception:
            pass

    # 3. Login to W&B without prompting if credentials exist
    if 'WANDB_API_KEY' in os.environ:
        wandb.login(key=os.environ['WANDB_API_KEY'], relogin=False)
    else:
        print("[W&B] Note: To avoid interactive prompt, name your secret 'WANDB_API_KEY' and paste full key into Value.")
        wandb.login(relogin=False)

    os.environ['WANDB_ENTITY'] = 'research-projekts'
    print('[W&B] Authenticated successfully. Target entity: research-projekts')
finally:
    if hasattr(signal, 'SIGALRM'):
        signal.alarm(0)


In [ ]:
# Step 2: Ingest Codebase & Dependencies
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("saros-grid-ssm"):
        os.system("git clone -q -b feature/initial-implementation https://github.com/jaimejarauta/saros-grid-ssm.git")
    if "saros-grid-ssm/src" not in sys.path:
        sys.path.insert(0, "saros-grid-ssm/src")

if "src" not in sys.path:
    sys.path.insert(0, "src")

try:
    from saros_grid_ssm.model import GridSpatioTemporalSSM
    from saros_grid_ssm.ree_data import generate_synthetic_ree_grid, REESpanishGridDataset
    from saros_grid_ssm.metrics import compute_wmape, compute_curtailment_f1, compute_grid_metrics
    print("[Ready] Successfully loaded Saros-Grid-SSM from package.")
except ImportError:
    print("[Fallback] Loading full exact standalone engine...")
    """Synthetic Spanish REE 50-Node Transmission Grid Data Generator and Loader.

    Models realistic 15-minute dispatch interval (96 timesteps/day) generation profiles,
    transmission line thermal capacities, regional renewable potential, and curtailment events.
    """

    from typing import Dict, List, Tuple
    import numpy as np
    import torch
    from torch.utils.data import Dataset

    SPANISH_50_NODES: List[str] = [
        # Madrid & Central Load Center (Nodes 0-3)
        "Madrid-Norte", "Madrid-Sur", "Madrid-Este", "Madrid-Oeste",
        # Andalusia - High Solar (Nodes 4-11)
        "Sevilla", "Cordoba", "Granada", "Cadiz", "Huelva", "Jaen", "Malaga", "Almeria",
        # Extremadura - High Solar (Nodes 12-13)
        "Caceres", "Badajoz",
        # Castile-La Mancha - Solar & Wind (Nodes 14-18)
        "Toledo", "Ciudad-Real", "Albacete", "Cuenca", "Guadalajara",
        # Aragon - High Wind (Nodes 19-22)
        "Zaragoza", "Huesca", "Teruel", "Ebro-Central",
        # Castile and Leon - Wind & Hydro (Nodes 23-31)
        "Valladolid", "Burgos", "Leon", "Palencia", "Salamanca", "Segovia", "Soria", "Zamora", "Avila",
        # Galicia - High Wind & Hydro (Nodes 32-35)
        "Coruna", "Lugo", "Ourense", "Pontevedra-Vigo",
        # Northern Spain / Basque / Cantabria / Asturias (Nodes 36-41)
        "Bilbao", "San-Sebastian", "Vitoria", "Pamplona", "Logrono", "Santander",
        # Catalonia - Heavy Load & Nuclear (Nodes 42-45)
        "Barcelona-Norte", "Barcelona-Sur", "Tarragona", "Girona",
        # Levante & Murcia (Nodes 46-49)
        "Valencia", "Alicante", "Castellon", "Murcia"
    ]

    def build_spanish_grid_adjacency(num_nodes: int = 50) -> np.ndarray:
        """Builds an unweighted symmetric adjacency matrix reflecting 400kV/220kV transmission trunk lines."""
        adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    
        # Transmission corridors: (node_i, node_j)
        corridors = [
            # Madrid internal ring and central links
            (0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3),
            (0, 18), (1, 14), (2, 18), (3, 31), (3, 28),
            # Andalusia internal network
            (4, 5), (4, 7), (4, 8), (5, 9), (6, 9), (6, 10), (6, 11), (7, 10), (10, 11),
            # Andalusia to Extremadura & Castile-La Mancha
            (4, 13), (5, 15), (9, 15), (11, 49),
            # Extremadura
            (12, 13), (12, 14), (13, 15), (12, 27),
            # Castile-La Mancha internal & connections
            (14, 15), (14, 17), (15, 16), (16, 17), (17, 18),
            (16, 46), (16, 49), (17, 21), (18, 29),
            # Aragon (Ebro Valley wind hub)
            (19, 20), (19, 21), (19, 22), (20, 22), (21, 22),
            (19, 39), (19, 44), (20, 40), (22, 44),
            # Castile and Leon internal network
            (23, 24), (23, 25), (23, 26), (23, 27), (23, 28), (23, 30), (23, 31),
            (24, 26), (24, 29), (24, 38), (25, 26), (25, 30), (25, 33),
            (27, 30), (28, 31), (29, 40),
            # Galicia
            (32, 33), (32, 35), (33, 34), (34, 35),
            (32, 41), (33, 25), (34, 30),
            # Northern Spain
            (36, 37), (36, 38), (36, 41), (37, 38), (37, 39), (38, 39), (39, 40),
            (41, 25),
            # Catalonia
            (42, 43), (42, 44), (42, 45), (43, 44),
            (44, 48), (44, 19),
            # Levante & Murcia
            (46, 47), (46, 48), (47, 49), (48, 44), (49, 11)
        ]
    
        for u, v in corridors:
            if u < num_nodes and v < num_nodes:
                adj[u, v] = 1.0
                adj[v, u] = 1.0

        return adj

    def normalize_adjacency(adj: np.ndarray) -> np.ndarray:
        """Computes symmetric normalized adjacency: D^{-1/2} (A + I) D^{-1/2}."""
        adj_loop = adj + np.eye(adj.shape[0], dtype=np.float32)
        degrees = np.sum(adj_loop, axis=1)
        d_inv_sqrt = np.power(degrees, -0.5)
        d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
        d_mat = np.diag(d_inv_sqrt)
        return d_mat @ adj_loop @ d_mat

    def generate_synthetic_ree_grid(
        num_days: int = 14,
        num_nodes: int = 50,
        seed: int = 2026
    ) -> Dict[str, np.ndarray]:
        """Generates synthetic 15-minute dispatch profiles for 50 Spanish REE transmission nodes."""
        rng = np.random.RandomState(seed)
        timesteps_per_day = 96
        total_timesteps = num_days * timesteps_per_day

        # 1. Base Capacities per Node (MW)
        # Solar capacity: high in South & Extremadura, low in North
        solar_cap = np.zeros(num_nodes, dtype=np.float32)
        # Wind capacity: high in Ebro (Aragon), Galicia, Castile & Leon
        wind_cap = np.zeros(num_nodes, dtype=np.float32)
        # Base peak demand (MW)
        base_demand = np.zeros(num_nodes, dtype=np.float32)
        # Transmission export line thermal limit (MW)
        export_limit = np.zeros(num_nodes, dtype=np.float32)

        for i in range(num_nodes):
            if i < 4:  # Madrid center: massive demand, low renewables
                solar_cap[i] = rng.uniform(50.0, 150.0)
                wind_cap[i] = rng.uniform(20.0, 80.0)
                base_demand[i] = rng.uniform(2500.0, 4200.0)
                export_limit[i] = rng.uniform(3000.0, 4500.0)
            elif 4 <= i <= 13:  # Andalusia & Extremadura: massive solar
                solar_cap[i] = rng.uniform(1800.0, 3500.0)
                wind_cap[i] = rng.uniform(300.0, 900.0)
                base_demand[i] = rng.uniform(600.0, 1400.0)
                export_limit[i] = rng.uniform(1200.0, 2000.0)  # Export bottleneck prone
            elif 14 <= i <= 18:  # Castile-La Mancha: solar + wind
                solar_cap[i] = rng.uniform(1200.0, 2400.0)
                wind_cap[i] = rng.uniform(800.0, 1800.0)
                base_demand[i] = rng.uniform(400.0, 900.0)
                export_limit[i] = rng.uniform(1400.0, 2200.0)
            elif 19 <= i <= 22:  # Aragon: high wind
                solar_cap[i] = rng.uniform(400.0, 900.0)
                wind_cap[i] = rng.uniform(2200.0, 4000.0)
                base_demand[i] = rng.uniform(500.0, 1200.0)
                export_limit[i] = rng.uniform(1500.0, 2400.0)  # Wind curtailment prone
            elif 23 <= i <= 31:  # Castile and Leon: wind & hydro
                solar_cap[i] = rng.uniform(500.0, 1100.0)
                wind_cap[i] = rng.uniform(1500.0, 3200.0)
                base_demand[i] = rng.uniform(400.0, 1000.0)
                export_limit[i] = rng.uniform(1600.0, 2500.0)
            elif 32 <= i <= 35:  # Galicia: high wind
                solar_cap[i] = rng.uniform(100.0, 400.0)
                wind_cap[i] = rng.uniform(1600.0, 2900.0)
                base_demand[i] = rng.uniform(600.0, 1300.0)
                export_limit[i] = rng.uniform(1300.0, 2100.0)
            elif 42 <= i <= 45:  # Catalonia: heavy load
                solar_cap[i] = rng.uniform(300.0, 700.0)
                wind_cap[i] = rng.uniform(400.0, 1000.0)
                base_demand[i] = rng.uniform(2000.0, 3800.0)
                export_limit[i] = rng.uniform(2500.0, 3800.0)
            else:  # Basque, Cantabria, Levante, Murcia
                solar_cap[i] = rng.uniform(400.0, 1200.0)
                wind_cap[i] = rng.uniform(400.0, 1200.0)
                base_demand[i] = rng.uniform(800.0, 1800.0)
                export_limit[i] = rng.uniform(1400.0, 2200.0)

        # 2. Time Array & Normalized Diurnal Profiles
        # 96 steps/day: step 0 = 00:00, step 28 = 07:00, step 56 = 14:00, step 84 = 21:00
        t_steps = np.arange(total_timesteps)
        day_step = t_steps % timesteps_per_day
        hour_of_day = (day_step * 15.0) / 60.0

        # Solar Diurnal Profile (Zero outside 07:00 - 21:00, peak at 14:00)
        solar_diurnal = np.maximum(0.0, np.sin(np.pi * (hour_of_day - 7.0) / 14.0))
        solar_diurnal[hour_of_day < 7.0] = 0.0
        solar_diurnal[hour_of_day > 21.0] = 0.0
        # Add non-linear parabolic zenith sharpness
        solar_diurnal = np.power(solar_diurnal, 1.3)

        # Wind Multi-Harmonic Profile (Nocturnal breeze + afternoon convection)
        wind_diurnal = (
            0.55
            + 0.25 * np.cos(2 * np.pi * (hour_of_day - 3.0) / 24.0)
            + 0.15 * np.sin(4 * np.pi * (hour_of_day - 15.0) / 24.0)
        )

        # Demand Profile (Bimodal: Morning peak 11:00, Evening peak 21:00, Night valley 05:00)
        demand_diurnal = (
            0.60
            + 0.25 * np.exp(-0.5 * ((hour_of_day - 11.5) / 2.5) ** 2)
            + 0.30 * np.exp(-0.5 * ((hour_of_day - 21.0) / 2.2) ** 2)
        )

        # Allocate feature matrices
        # in_features: [solar_potential, wind_potential, demand, export_limit]
        inputs = np.zeros((total_timesteps, num_nodes, 4), dtype=np.float32)
        # injection target: [actual_solar_mw, actual_wind_mw]
        injections = np.zeros((total_timesteps, num_nodes, 2), dtype=np.float32)
        # curtailment risk: [bottleneck_indicator]
        curtailments = np.zeros((total_timesteps, num_nodes, 1), dtype=np.float32)

        # Weather front progression across days
        weather_phase = rng.uniform(0, 2 * np.pi, size=(num_nodes,))

        for t in range(total_timesteps):
            day_idx = t // timesteps_per_day
            # Synoptic daily weather variations
            weather_cloud = 0.85 + 0.15 * np.sin(2 * np.pi * (day_idx / 7.0) + weather_phase)
            weather_wind_gust = 0.90 + 0.25 * np.cos(2 * np.pi * (day_idx / 5.0) + weather_phase)

            sol_pot = solar_cap * solar_diurnal[t] * weather_cloud
            sol_pot += rng.normal(0.0, 5.0, size=(num_nodes,))
            sol_pot = np.maximum(0.0, sol_pot)

            wnd_pot = wind_cap * wind_diurnal[t] * weather_wind_gust
            wnd_pot += rng.normal(0.0, 10.0, size=(num_nodes,))
            wnd_pot = np.maximum(0.0, wnd_pot)

            dem = base_demand * demand_diurnal[t] * (1.0 + rng.normal(0.0, 0.02, size=(num_nodes,)))

            inputs[t, :, 0] = sol_pot
            inputs[t, :, 1] = wnd_pot
            inputs[t, :, 2] = dem
            inputs[t, :, 3] = export_limit

            # Physical Grid Bottleneck & Curtailment calculation
            # Net nodal generation excess
            total_ren = sol_pot + wnd_pot
            net_excess = total_ren - dem - export_limit

            is_curtailed = net_excess > 0.0
            curtailments[t, :, 0] = is_curtailed.astype(np.float32)

            # Curtailed dispatch: excess generation cannot be injected
            curtailed_solar = np.where(is_curtailed, sol_pot - 0.6 * net_excess, sol_pot)
            curtailed_wind = np.where(is_curtailed, wnd_pot - 0.4 * net_excess, wnd_pot)

            injections[t, :, 0] = np.maximum(0.0, curtailed_solar)
            injections[t, :, 1] = np.maximum(0.0, curtailed_wind)

        adj = build_spanish_grid_adjacency(num_nodes)
        adj_norm = normalize_adjacency(adj)

        return {
            "inputs": inputs,
            "injections": injections,
            "curtailments": curtailments,
            "adjacency": adj,
            "adjacency_norm": adj_norm,
            "node_names": SPANISH_50_NODES[:num_nodes],
        }

    class REESpanishGridDataset(Dataset):
        """PyTorch Dataset yielding 24-hour (96-timestep) day-ahead grid dispatch windows."""
        def __init__(
            self,
            inputs: np.ndarray,
            injections: np.ndarray,
            curtailments: np.ndarray,
            window_size: int = 96,
            stride: int = 96
        ):
            self.inputs = torch.from_numpy(inputs).float()
            self.injections = torch.from_numpy(injections).float()
            self.curtailments = torch.from_numpy(curtailments).float()
            self.window_size = window_size
            self.stride = stride

            # Number of non-overlapping or strided daily windows
            self.total_timesteps = inputs.shape[0]
            self.indices = list(range(0, self.total_timesteps - window_size + 1, stride))

        def __len__(self) -> int:
            return len(self.indices)

        def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
            start = self.indices[idx]
            end = start + self.window_size
            x_window = self.inputs[start:end]              # (96, num_nodes, 4)
            y_inj_window = self.injections[start:end]      # (96, num_nodes, 2)
            y_curt_window = self.curtailments[start:end]   # (96, num_nodes, 1)
            return x_window, y_inj_window, y_curt_window


    """Spatio-Temporal Selective State Space Model for Transmission Grid Forecasting.

    Combines non-Euclidean graph spatial diffusion with continuous-time diagonal selective
    state space recurrence for nodal renewable injection and curtailment bottleneck prediction.
    """

    from typing import Dict, Optional, Tuple
    import math
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class GraphSpatialDiffusion(nn.Module):
        """K-hop graph spatial diffusion operator reflecting physical transmission power flow."""
        def __init__(self, d_in: int, d_out: int, num_hops: int = 2):
            super().__init__()
            self.num_hops = num_hops
            self.d_in = d_in
            self.d_out = d_out
        
            # Learnable filter weights for each spatial hop k in {0, ..., num_hops}
            self.hop_weights = nn.ParameterList([
                nn.Parameter(torch.empty(d_in, d_out)) for _ in range(num_hops + 1)
            ])
            self.bias = nn.Parameter(torch.zeros(d_out))
            self.norm = nn.LayerNorm(d_out)
            self.reset_parameters()

        def reset_parameters(self):
            for w in self.hop_weights:
                nn.init.xavier_uniform_(w)
            nn.init.zeros_(self.bias)

        def forward(self, x: torch.Tensor, adj_norm: torch.Tensor) -> torch.Tensor:
            """Forward pass for spatial diffusion.
        
            Args:
                x: Node features of shape (B, T, N, D_in)
                adj_norm: Normalized symmetric adjacency matrix (N, N)
            Returns:
                Diffused features of shape (B, T, N, D_out)
            """
            # P_0 = X
            p_curr = x
            out = torch.matmul(p_curr, self.hop_weights[0])

            # Successive Chebyshev / Adjacency polynomial diffusion hops
            for k in range(1, self.num_hops + 1):
                # p_k = A_norm * p_{k-1} along the node dimension N
                # (N, N) @ (B, T, N, D_in) -> (B, T, N, D_in)
                p_next = torch.einsum("nm,btmd->btnd", adj_norm, p_curr)
                out = out + torch.matmul(p_next, self.hop_weights[k])
                p_curr = p_next

            out = out + self.bias
            return self.norm(F.gelu(out))

    class SelectiveSSMBlock(nn.Module):
        """Selective continuous-time diagonal state space recurrence layer.
    
        Implements input-dependent time-step discretization (Mamba/S4D formulation):
            A_c = -exp(alpha)  (BIBO stable diagonal transition)
            Delta_t = softplus(W_delta x_t + b_delta)
            B_t = W_B x_t,  C_t = W_C x_t
            h_t = exp(Delta_t A_c) * h_{t-1} + (Delta_t B_t) * x_t
            y_t = C_t^T h_t + D x_t
        """
        def __init__(
            self,
            d_model: int,
            d_state: int = 16,
            bidirectional: bool = True,
            dropout: float = 0.05
        ):
            super().__init__()
            self.d_model = d_model
            self.d_state = d_state
            self.bidirectional = bidirectional

            # Continuous diagonal transition matrix A_c parameterization
            # log_alpha enforces Re(A_c) = -exp(log_alpha) < 0 for unconditional stability
            self.log_alpha_fwd = nn.Parameter(torch.linspace(0.1, 2.0, d_state).unsqueeze(0))
            if bidirectional:
                self.log_alpha_bwd = nn.Parameter(torch.linspace(0.1, 2.0, d_state).unsqueeze(0))

            # Input-dependent selective projections (Delta, B, C)
            self.proj_delta_fwd = nn.Linear(d_model, d_state)
            self.proj_b_fwd = nn.Linear(d_model, d_state)
            self.proj_c_fwd = nn.Linear(d_model, d_state)

            if bidirectional:
                self.proj_delta_bwd = nn.Linear(d_model, d_state)
                self.proj_b_bwd = nn.Linear(d_model, d_state)
                self.proj_c_bwd = nn.Linear(d_model, d_state)

            # Feedthrough skip connection D
            self.d_skip = nn.Parameter(torch.ones(d_model))

            # Output projection and normalization
            out_dim = d_model * 2 if bidirectional else d_model
            self.out_proj = nn.Linear(out_dim, d_model)
            self.norm = nn.LayerNorm(d_model)
            self.dropout = nn.Dropout(dropout)

        def _selective_scan(
            self,
            x: torch.Tensor,
            log_alpha: torch.Tensor,
            proj_delta: nn.Linear,
            proj_b: nn.Linear,
            proj_c: nn.Linear,
            reverse: bool = False
        ) -> torch.Tensor:
            """Performs selective continuous-time SSM scan along sequence length dimension.
        
            Args:
                x: Tensor of shape (BatchSize, SeqLen, D_model)
            Returns:
                Output tensor of shape (BatchSize, SeqLen, D_model)
            """
            b_size, seq_len, _ = x.shape
            device = x.device

            # Continuous transition A_c = -exp(log_alpha) (1, d_state)
            a_cont = -torch.exp(log_alpha)

            # Precompute input-dependent projections
            delta = F.softplus(proj_delta(x))   # (B, T, d_state)
            b_t = proj_b(x)                     # (B, T, d_state)
            c_t = proj_c(x)                     # (B, T, d_state)

            # Discretize: A_bar = exp(delta * A_c), B_bar = delta * B_t
            a_bar = torch.exp(delta * a_cont.unsqueeze(0))  # (B, T, d_state)
            b_bar = delta * b_t                            # (B, T, d_state)

            h_state = torch.zeros(b_size, self.d_state, device=device, dtype=x.dtype)
            y_steps = []

            step_indices = range(seq_len - 1, -1, -1) if reverse else range(seq_len)
            for t in step_indices:
                # Scalar channel modulation: take mean across d_model for state input
                x_step_input = x[:, t, :].mean(dim=-1, keepdim=True)
                # h_t = A_bar * h_{t-1} + B_bar * x_t
                h_state = a_bar[:, t, :] * h_state + b_bar[:, t, :] * x_step_input
                # y_t = C_t * h_t
                y_state = h_state * c_t[:, t, :]  # (B, d_state)
                y_steps.append(y_state)

            if reverse:
                y_steps = y_steps[::-1]

            y_stacked = torch.stack(y_steps, dim=1)  # (B, T, d_state)
            # Tile or project to d_model
            if self.d_state != self.d_model:
                # Linear expansion to d_model
                rep_factor = math.ceil(self.d_model / self.d_state)
                y_out = y_stacked.repeat(1, 1, rep_factor)[:, :, :self.d_model]
            else:
                y_out = y_stacked

            return y_out

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            """Forward pass for selective temporal SSM.
        
            Args:
                x: Input tensor of shape (B, T, N, D_model)
            Returns:
                Output tensor of shape (B, T, N, D_model)
            """
            b, t, n, d = x.shape
            # Flatten spatial nodes into batch: (B * N, T, D)
            x_flat = x.permute(0, 2, 1, 3).reshape(b * n, t, d)

            # Forward selective scan
            y_fwd = self._selective_scan(
                x_flat, self.log_alpha_fwd, self.proj_delta_fwd, self.proj_b_fwd, self.proj_c_fwd, reverse=False
            )

            if self.bidirectional:
                # Backward selective scan
                y_bwd = self._selective_scan(
                    x_flat, self.log_alpha_bwd, self.proj_delta_bwd, self.proj_b_bwd, self.proj_c_bwd, reverse=True
                )
                y_combined = torch.cat([y_fwd, y_bwd], dim=-1)
            else:
                y_combined = y_fwd

            # Project back to d_model and add skip connection
            out = self.out_proj(y_combined) + x_flat * self.d_skip
            out = self.dropout(out)
            out = self.norm(out)

            # Reshape back to (B, T, N, D_model)
            return out.reshape(b, n, t, d).permute(0, 2, 1, 3)

    class GridForecastingHead(nn.Module):
        """Dual forecasting head: continuous renewable MW injection and curtailment probability."""
        def __init__(self, d_model: int, out_features: int = 2):
            super().__init__()
            # Continuous injection head (Solar and Wind MW)
            self.injection_head = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.GELU(),
                nn.Linear(d_model, out_features),
                nn.Softplus()  # Enforces non-negative MW injection
            )
            # Discrete curtailment bottleneck classification head
            self.curtailment_head = nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.GELU(),
                nn.Linear(d_model // 2, 1),
                nn.Sigmoid()  # Probability in [0, 1]
            )

        def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
            """Maps hidden representations to dispatch outputs.
        
            Args:
                h: Hidden state of shape (B, T, N, D_model)
            Returns:
                injection_mw: (B, T, N, 2)
                curtailment_prob: (B, T, N, 1)
            """
            injection = self.injection_head(h)
            curtailment = self.curtailment_head(h)
            return injection, curtailment

    class GridSpatioTemporalSSM(nn.Module):
        """Spatio-Temporal Selective State Space Model for Transmission Grid Dispatch."""
        def __init__(
            self,
            in_features: int = 4,
            out_features: int = 2,
            d_model: int = 64,
            d_state: int = 16,
            num_spatial_hops: int = 2,
            num_layers: int = 2,
            bidirectional: bool = True,
            dropout: float = 0.05
        ):
            super().__init__()
            self.in_proj = nn.Linear(in_features, d_model)
            self.num_layers = num_layers

            self.spatial_layers = nn.ModuleList([
                GraphSpatialDiffusion(d_model, d_model, num_hops=num_spatial_hops)
                for _ in range(num_layers)
            ])
            self.temporal_layers = nn.ModuleList([
                SelectiveSSMBlock(d_model, d_state=d_state, bidirectional=bidirectional, dropout=dropout)
                for _ in range(num_layers)
            ])

            self.head = GridForecastingHead(d_model, out_features=out_features)

        def forward(
            self,
            x: torch.Tensor,
            adj_norm: torch.Tensor
        ) -> Dict[str, torch.Tensor]:
            """Full forward pass for 15-minute dispatch forecasting.
        
            Args:
                x: Input tensor of shape (B, T, N, in_features)
                adj_norm: Normalized graph adjacency matrix of shape (N, N)
            Returns:
                Dictionary containing:
                    "injection_mw": Predicted renewable injections (B, T, N, 2)
                    "curtailment_prob": Predicted bottleneck probability (B, T, N, 1)
            """
            h = self.in_proj(x)

            for s_layer, t_layer in zip(self.spatial_layers, self.temporal_layers):
                # 1. Spatial diffusion across transmission corridors with residual
                h = h + s_layer(h, adj_norm)
                # 2. Selective temporal state space recurrence with residual
                h = h + t_layer(h)

            inj_pred, curt_pred = self.head(h)

            return {
                "injection_mw": inj_pred,
                "curtailment_prob": curt_pred
            }


    """Evaluation metrics for REE grid renewable injection forecasting and curtailment detection."""

    from typing import Dict, Union
    import numpy as np
    import torch

    def compute_wmape(
        y_true: Union[torch.Tensor, np.ndarray],
        y_pred: Union[torch.Tensor, np.ndarray],
        eps: float = 1e-5
    ) -> float:
        """Weighted Mean Absolute Percentage Error (wMAPE).
    
        Formula: sum(|y_true - y_pred|) / (sum(|y_true|) + eps)
        """
        if isinstance(y_true, torch.Tensor):
            diff = torch.abs(y_true - y_pred).sum()
            denom = torch.abs(y_true).sum() + eps
            return (diff / denom).item()
        else:
            diff = np.sum(np.abs(y_true - y_pred))
            denom = np.sum(np.abs(y_true)) + eps
            return float(diff / denom)

    def compute_curtailment_f1(
        y_true: Union[torch.Tensor, np.ndarray],
        y_pred_prob: Union[torch.Tensor, np.ndarray],
        threshold: float = 0.5,
        eps: float = 1e-7
    ) -> Dict[str, float]:
        """Precision, recall, and F1 score for curtailment bottleneck event classification."""
        if isinstance(y_true, torch.Tensor):
            y_true_np = (y_true > 0.5).detach().cpu().numpy().astype(int).flatten()
        else:
            y_true_np = (y_true > 0.5).astype(int).flatten()

        if isinstance(y_pred_prob, torch.Tensor):
            y_pred_np = (y_pred_prob >= threshold).detach().cpu().numpy().astype(int).flatten()
        else:
            y_pred_np = (y_pred_prob >= threshold).astype(int).flatten()

        tp = np.sum((y_true_np == 1) & (y_pred_np == 1))
        fp = np.sum((y_true_np == 0) & (y_pred_np == 1))
        fn = np.sum((y_true_np == 1) & (y_pred_np == 0))
        tn = np.sum((y_true_np == 0) & (y_pred_np == 0))

        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2 * (precision * recall) / (precision + recall + eps)
        accuracy = (tp + tn) / max(1, len(y_true_np))

        return {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "accuracy": float(accuracy),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        }

    def compute_grid_metrics(
        injection_true: Union[torch.Tensor, np.ndarray],
        injection_pred: Union[torch.Tensor, np.ndarray],
        curtailment_true: Union[torch.Tensor, np.ndarray],
        curtailment_pred_prob: Union[torch.Tensor, np.ndarray],
        curtail_threshold: float = 0.5,
    ) -> Dict[str, float]:
        """Computes joint telemetry for injection wMAPE, MAE, RMSE and curtailment classification."""
        wmape = compute_wmape(injection_true, injection_pred)
    
        if isinstance(injection_true, torch.Tensor):
            mae = torch.mean(torch.abs(injection_true - injection_pred)).item()
            rmse = torch.sqrt(torch.mean((injection_true - injection_pred) ** 2)).item()
        else:
            mae = float(np.mean(np.abs(injection_true - injection_pred)))
            rmse = float(np.sqrt(np.mean((injection_true - injection_pred) ** 2)))

        curtail_metrics = compute_curtailment_f1(
            curtailment_true, curtailment_pred_prob, threshold=curtail_threshold
        )

        return {
            "wmape": float(wmape),
            "mae": float(mae),
            "rmse": float(rmse),
            "curtailment_f1": curtail_metrics["f1"],
            "curtailment_precision": curtail_metrics["precision"],
            "curtailment_recall": curtail_metrics["recall"],
            "curtailment_accuracy": curtail_metrics["accuracy"],
        }

    print("[Ready] Loaded exact standalone Saros-Grid-SSM components.")


In [ ]:
# Step 3: Generate REE 50-Node Grid & Initialize Model
seed = 2026
torch.manual_seed(seed)
num_nodes = 50

raw_data = generate_synthetic_ree_grid(num_days=21, num_nodes=num_nodes, seed=seed)
adj_norm = torch.from_numpy(raw_data["adjacency_norm"]).float().to(device)

train_split = 14 * 96
train_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][:train_split],
    injections=raw_data["injections"][:train_split],
    curtailments=raw_data["curtailments"][:train_split],
    window_size=96,
    stride=48
)
val_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][train_split:],
    injections=raw_data["injections"][train_split:],
    curtailments=raw_data["curtailments"][train_split:],
    window_size=96,
    stride=96
)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

model = GridSpatioTemporalSSM(
    in_features=4,
    out_features=2,
    d_model=64,
    d_state=16,
    num_spatial_hops=2,
    num_layers=2,
    bidirectional=True,
    dropout=0.05
).to(device)

print(f"Initialized GridSpatioTemporalSSM ({sum(p.numel() for p in model.parameters()):,} params)")


In [ ]:
# Step 4: Run Smoke Training Loop (15-min dispatch with p.u. scaling)
import torch.nn as nn

s_base = 1000.0
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
steps, max_steps = 0, 300
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
huber_loss_fn = nn.SmoothL1Loss(beta=0.01)

p_curt = float(raw_data["curtailments"][:train_split].mean())
pos_weight = torch.tensor([(1.0 - p_curt) / max(p_curt, 1e-6)], device=device)
bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model.train()
t0 = time.time()

while steps < max_steps:
    for x_b, y_inj_b, y_curt_b in train_loader:
        x_b, y_inj_b, y_curt_b = x_b.to(device), y_inj_b.to(device), y_curt_b.to(device)
        x_b_pu = x_b / s_base
        y_inj_pu = y_inj_b / s_base

        optimizer.zero_grad()
        preds = model(x_b_pu, adj_norm)
        loss = huber_loss_fn(preds["injection_mw"], y_inj_pu) + 0.10 * bce_loss_fn(preds["curtailment_logits"], y_curt_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        steps += 1
        if steps % 25 == 0:
            pred_mw_step = preds["injection_mw"] * s_base
            wmape_step = compute_wmape(y_inj_b, pred_mw_step)
            print(f"Step {steps:03d}/{max_steps} | Loss: {loss.item():.4f} | Batch wMAPE: {wmape_step * 100:.2f}%")
        if steps >= max_steps:
            break

print(f"Training elapsed: {time.time() - t0:.2f}s")


In [ ]:
# Step 5: Evaluate Pre-Registered Gate Criteria
model.eval()
with torch.no_grad():
    for x_val, y_inj_val, y_curt_val in val_loader:
        x_val, y_inj_val, y_curt_val = x_val.to(device), y_inj_val.to(device), y_curt_val.to(device)
        x_val_pu = x_val / s_base
        preds_val = model(x_val_pu, adj_norm)
        pred_mw_val = preds_val["injection_mw"] * s_base
        metrics = compute_grid_metrics(
            y_inj_val, pred_mw_val,
            y_curt_val, preds_val["curtailment_prob"]
        )

wmape_pct = metrics["wmape"] * 100.0
gate_passed = metrics["wmape"] < 0.065 and metrics["curtailment_f1"] >= 0.75

print("=" * 60)
print(f"Day-Ahead Injection wMAPE : {wmape_pct:.2f}% (Target: < 6.50%)")
print(f"Curtailment F1 Score      : {metrics['curtailment_f1']:.4f} (Target: >= 0.750)")
print(f"Pre-Registered Gate 1     : {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 5b: 300-Step Spanish REE Transmission Grid Dispatch Benchmark
import os, sys, subprocess

if 'google.colab' in sys.modules:
    repo_dir = '/content/saros-grid-ssm'
    if not os.path.exists(repo_dir):
        print('Cloning saros-grid-ssm (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/saros-grid-ssm.git', repo_dir], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo_dir, 'wandb'], check=True)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
        sys.path.insert(0, f"{repo_dir}/src")
    os.chdir(repo_dir)

!python /content/saros-grid-ssm/scripts/run_smoke.py --steps 300 --wandb --wandb-project research-projekts --wandb-mode online --wandb-name saros-grid-ssm-l4


In [ ]:
# Step Pre-Kill: Explicit W&B Telemetry Cloud Upload
# Ensures every run (online or offline) is completely synced and uploaded to W&B cloud before teardown
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true
!wandb sync --include-offline /tmp/wandb wandb 2>/dev/null || true


In [ ]:
# Step Final: Credit Protection & Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
# Set to False by default so the virtual machine stays alive after execution
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
else:
    print("\n[Credit Guard] AUTO_KILL_ON_FINISH is False. Keeping VM instance alive for inspection.")
